## 1. Load Dataset

We begin by loading the first 5000 rows of the web server log data, focusing only on the `request` and `http_user_agent` columns. Missing values are removed to ensure clean input for tokenization.


### 📥 Download Dataset

Please download the dataset manually from the following Google Drive link and place it in the same directory as the notebook before running:

🔗 [Download web_logs.csv](https://drive.google.com/file/d/1oa-D-CmnDOEUqyaQRisDZJAFNsuaIJT0/view?usp=drive_link)


In [374]:
import pandas as pd

# Load only 5000 rows with the two needed columns
df = pd.read_csv("log_data.csv", usecols=["request", "http_user_agent"], nrows=5000)

# Drop rows with missing values in either column
df_nlp = df.dropna(subset=["request", "http_user_agent"]).copy()

print(df_nlp.shape)  


(5000, 2)


In [375]:

df_nlp.head()


,http_user_agent,request
0,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/qH48WEobbNgDtf...
1,FreshpingBot/1.0 (+https://freshping.io/),GET / HTTP/1.1
2,WordPress/6.7.1; https://prophaze.com,POST /wp-cron.php?doing_wp_cron=1735603198.430...
3,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/pVcVfAxQguStY2...
4,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/R8tx5MFoIw5Nmq...


## 2. Tokenizing the `request` Field

Each HTTP request string is split into three parts:
- **Method** (e.g., GET, POST)
- **Path** (e.g., `/about/contact`)
- **Query Parameters** (e.g., `id=1&ref=abc`)

We extract:
- `method`, `version`
- `path_tokens`: tokenized segments of the path
- `query_tokens`: tokenized query parameters

In [376]:
#Tokenizing request
def tokenize_request_full(request):
    try:
        method, path_query, version = request.strip().split()
        path = path_query.split('?')[0]
        query = path_query.split('?')[1] if '?' in path_query else ''
        method_token = method
        version_token = version
        path_tokens = path.strip('/').split('/') if path else []
        query_tokens = query.split('&') if query else []
        return {
            "method": method_token,
            "version": version_token,
            "path_tokens": path_tokens,
            "query_tokens": query_tokens
        }
    except Exception as e:
        return {
            "method": None,
            "version": None,
            "path_tokens": [],
            "query_tokens": []
        }


In [377]:
# Apply tokenizer to 'request' column
df_req_tokens = df_nlp['request'].astype(str).apply(tokenize_request_full).apply(pd.Series)

# Merge into main DataFrame
df_nlp = pd.concat([df_nlp, df_req_tokens], axis=1)

# Results
df_nlp[['request', 'method', 'version', 'path_tokens', 'query_tokens']].head()


,request,method,version,path_tokens,query_tokens
0,GET /.well-known/acme-challenge/qH48WEobbNgDtf...,GET,HTTP/1.1,"[.well-known, acme-challenge, qH48WEobbNgDtfCG...",[]
1,GET / HTTP/1.1,GET,HTTP/1.1,[],[]
2,POST /wp-cron.php?doing_wp_cron=1735603198.430...,POST,HTTP/1.1,[wp-cron.php],[doing_wp_cron=1735603198.4308490753173828125000]
3,GET /.well-known/acme-challenge/pVcVfAxQguStY2...,GET,HTTP/1.1,"[.well-known, acme-challenge, pVcVfAxQguStY2CJ...",[]
4,GET /.well-known/acme-challenge/R8tx5MFoIw5Nmq...,GET,HTTP/1.1,"[.well-known, acme-challenge, R8tx5MFoIw5NmqeC...",[]


## 3. Parsing the `http_user_agent` Field

User-Agent strings are parsed to extract:
- **Browser** (e.g., Chrome, Firefox)
- **Operating System** (e.g., ios, Android)
- **Device Type** (Desktop, Mobile, Tablet)
- **Automation Indicator** (whether the request came from a bot/crawler)


In [378]:
pip install pyyaml ua-parser user-agents


Note: you may need to restart the kernel to use updated packages.


In [379]:

from user_agents import parse

# Custom parsing function
def parse_user_agent_custom(ua_string):
    ua = parse(ua_string)

    # Detect automation (bot)
    automation = "Yes" if ua.is_bot else "No"

    # Extract browser and OS
    browser = ua.browser.family or "other"
    os = ua.os.family or "other"

    # Detect device type
    if ua.is_mobile:
        device = "mobile"
    elif ua.is_tablet:
        device = "tablet"
    else:
        device = "desktop"

    return pd.Series([browser.lower(), os.lower(), device.lower(), automation.lower()])

# Apply the function on actual df_nlp
df_nlp[['browser', 'os', 'device', 'automation']] = df_nlp['http_user_agent'].astype(str).apply(parse_user_agent_custom)

df_nlp[['http_user_agent', 'browser', 'os', 'device', 'automation']].tail(10)


,http_user_agent,browser,os,device,automation
4990,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4991,WP Rocket/Preload Mozilla/5.0 (iPhone; CPU iPh...,mobile safari,ios,mobile,no
4992,WP Rocket/Preload,other,other,desktop,no
4993,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4994,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4995,WP Rocket/Preload Mozilla/5.0 (iPhone; CPU iPh...,mobile safari,ios,mobile,no
4996,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4997,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4998,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4999,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no


In [380]:
df_nlp[['http_user_agent', 'browser', 'os', 'device', 'automation']].head()


,http_user_agent,browser,os,device,automation
0,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
1,FreshpingBot/1.0 (+https://freshping.io/),freshpingbot,other,desktop,yes
2,WordPress/6.7.1; https://prophaze.com,wordpress,other,desktop,yes
3,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no
4,jetstack-cert-manager/v0.16.1 (clean),other,other,desktop,no


In [381]:
df_nlp['browser'] = df_nlp['browser'].astype(str).str.lower()
df_nlp['os'] = df_nlp['os'].astype(str).str.lower()
df_nlp['device'] = df_nlp['device'].astype(str).str.lower()
df_nlp['method'] = df_nlp['method'].astype(str).str.lower()



In [382]:
df_nlp['automation'] = df_nlp['automation'].map({'no': 0, 'yes': 1})


In [383]:
#Converts values like 'HTTP/1.1' → '1.1' by using a regular expression
df_nlp['version'] = df_nlp['version'].str.extract(r'HTTP/(\d\.\d)')


In [384]:
df_nlp.head()

,http_user_agent,request,method,version,path_tokens,query_tokens,browser,os,device,automation
0,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/qH48WEobbNgDtf...,get,1.1,"[.well-known, acme-challenge, qH48WEobbNgDtfCG...",[],other,other,desktop,0
1,FreshpingBot/1.0 (+https://freshping.io/),GET / HTTP/1.1,get,1.1,[],[],freshpingbot,other,desktop,1
2,WordPress/6.7.1; https://prophaze.com,POST /wp-cron.php?doing_wp_cron=1735603198.430...,post,1.1,[wp-cron.php],[doing_wp_cron=1735603198.4308490753173828125000],wordpress,other,desktop,1
3,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/pVcVfAxQguStY2...,get,1.1,"[.well-known, acme-challenge, pVcVfAxQguStY2CJ...",[],other,other,desktop,0
4,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/R8tx5MFoIw5Nmq...,get,1.1,"[.well-known, acme-challenge, R8tx5MFoIw5NmqeC...",[],other,other,desktop,0


## 4. Label Encoding Categorical Features

To prepare categorical features for machine learning models, we apply `LabelEncoder` from `sklearn` to the following columns:
- `method`
- `version`
- `browser`
- `os`
- `device`

Each unique category in these columns is mapped to an integer. The encoded columns are added to the DataFrame with the suffix `_enc`.

This step helps convert non-numeric values into numerical format, which is essential for most ML models.


In [385]:
from sklearn.preprocessing import LabelEncoder

# Standardize column names (just in case)
df_nlp.columns = df_nlp.columns.str.strip().str.lower()

# Define the fields to encode
categorical_cols = ['method', 'browser', 'os', 'device']

# Create a dictionary to store label encoders for each column
label_encoders = {}

# Apply Label Encoding
for col in categorical_cols:
    if col in df_nlp.columns:
        le = LabelEncoder()
        df_nlp[col + '_enc'] = le.fit_transform(df_nlp[col].astype(str))
        label_encoders[col] = le
        print(f"Encoded '{col}' → '{col}_enc'")
    else:
        print(f"Column '{col}' not found.")


Encoded 'method' → 'method_enc'
Encoded 'browser' → 'browser_enc'
Encoded 'os' → 'os_enc'
Encoded 'device' → 'device_enc'


In [386]:
df_nlp.head()

,http_user_agent,request,method,version,path_tokens,query_tokens,browser,os,device,automation,method_enc,browser_enc,os_enc,device_enc
0,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/qH48WEobbNgDtf...,get,1.1,"[.well-known, acme-challenge, qH48WEobbNgDtfCG...",[],other,other,desktop,0,0,24,5,0
1,FreshpingBot/1.0 (+https://freshping.io/),GET / HTTP/1.1,get,1.1,[],[],freshpingbot,other,desktop,1,0,15,5,0
2,WordPress/6.7.1; https://prophaze.com,POST /wp-cron.php?doing_wp_cron=1735603198.430...,post,1.1,[wp-cron.php],[doing_wp_cron=1735603198.4308490753173828125000],wordpress,other,desktop,1,2,33,5,0
3,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/pVcVfAxQguStY2...,get,1.1,"[.well-known, acme-challenge, pVcVfAxQguStY2CJ...",[],other,other,desktop,0,0,24,5,0
4,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/R8tx5MFoIw5Nmq...,get,1.1,"[.well-known, acme-challenge, R8tx5MFoIw5NmqeC...",[],other,other,desktop,0,0,24,5,0


In [387]:
!pip install tensorflow


## 5. Rare Token Handling and Sequence Encoding

To reduce noise and improve generalization, we first identify and replace infrequent tokens in both `path_tokens` and `query_tokens` using a placeholder (`__RARE__`). After cleaning, we tokenize and pad the sequences.

### Steps:

1. **Rare Token Replacement**:
   - Flatten all path and query tokens and count their frequency.
   - Replace tokens that occur less than a specified threshold (`RARE_THRESHOLD = 3`) with `'__RARE__'`.

2. **Token String Conversion**:
   - Join cleaned token lists into space-separated strings (`path_str`, `query_str`) to prepare for vectorization.

3. **Tokenization and Padding**:
   - A reusable function `tokenize_and_pad()` is defined to:
     - Fit a tokenizer on the text
     - Convert text to integer sequences
     - Pad sequences to a fixed length (`maxlen=10`)
   - Applied separately for both `path_str` and `query_str`.

This optimized approach avoids redundancy, ensures consistency in preprocessing, and prepares the data for embedding or downstream deep learning tasks.


In [388]:
#encoding path_str and query_str
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Flatten tokens to count frequency
all_path_tokens = [token for tokens in df_nlp['path_tokens'] for token in tokens]
all_query_tokens = [token for tokens in df_nlp['query_tokens'] for token in tokens]
token_counts = Counter(all_path_tokens + all_query_tokens)
RARE_THRESHOLD = 3

# 2. Replace rare tokens with '__RARE__'
def replace_rare(tokens, counts, threshold=RARE_THRESHOLD):
    return [token if counts[token] >= threshold else '__RARE__' for token in tokens]

df_nlp['path_tokens'] = df_nlp['path_tokens'].apply(lambda x: replace_rare(x, token_counts))
df_nlp['query_tokens'] = df_nlp['query_tokens'].apply(lambda x: replace_rare(x, token_counts))

# 3. Convert tokens to strings for encoding
df_nlp['path_str'] = df_nlp['path_tokens'].apply(lambda x: ' '.join(x))
df_nlp['query_str'] = df_nlp['query_tokens'].apply(lambda x: ' '.join(x))

# 4. Tokenize and pad
def tokenize_and_pad(text_series, maxlen=10):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(text_series)
    seq = tokenizer.texts_to_sequences(text_series)
    padded = pad_sequences(seq, maxlen=maxlen, padding='post')
    return seq, padded, tokenizer

# Apply for path and query
df_nlp['path_seq'], path_seq_padded, path_tokenizer = tokenize_and_pad(df_nlp['path_str'])
df_nlp['query_seq'], query_seq_padded, query_tokenizer = tokenize_and_pad(df_nlp['query_str'])


In [389]:
df_nlp.head()

,http_user_agent,request,method,version,path_tokens,query_tokens,browser,os,device,automation,method_enc,browser_enc,os_enc,device_enc,path_str,query_str,path_seq,query_seq
0,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/qH48WEobbNgDtf...,get,1.1,"[.well-known, acme-challenge, qH48WEobbNgDtfCG...",[],other,other,desktop,0,0,24,5,0,.well-known acme-challenge qH48WEobbNgDtfCGyIu...,,"[1, 2, 3, 4, 10]",[]
1,FreshpingBot/1.0 (+https://freshping.io/),GET / HTTP/1.1,get,1.1,[],[],freshpingbot,other,desktop,1,0,15,5,0,,,[],[]
2,WordPress/6.7.1; https://prophaze.com,POST /wp-cron.php?doing_wp_cron=1735603198.430...,post,1.1,[wp-cron.php],[__RARE__],wordpress,other,desktop,1,2,33,5,0,wp-cron.php,__RARE__,"[11, 26, 12]",[1]
3,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/pVcVfAxQguStY2...,get,1.1,"[.well-known, acme-challenge, pVcVfAxQguStY2CJ...",[],other,other,desktop,0,0,24,5,0,.well-known acme-challenge pVcVfAxQguStY2CJnKL...,,"[1, 2, 3, 4, 8, 9]",[]
4,jetstack-cert-manager/v0.16.1 (clean),GET /.well-known/acme-challenge/R8tx5MFoIw5Nmq...,get,1.1,"[.well-known, acme-challenge, R8tx5MFoIw5NmqeC...",[],other,other,desktop,0,0,24,5,0,.well-known acme-challenge R8tx5MFoIw5NmqeCtHb...,,"[1, 2, 3, 4, 7]",[]


In [390]:
df_nlp.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   http_user_agent  5000 non-null   object
 1   request          5000 non-null   object
 2   method           5000 non-null   object
 3   version          5000 non-null   object
 4   path_tokens      5000 non-null   object
 5   query_tokens     5000 non-null   object
 6   browser          5000 non-null   object
 7   os               5000 non-null   object
 8   device           5000 non-null   object
 9   automation       5000 non-null   int64 
 10  method_enc       5000 non-null   int32 
 11  browser_enc      5000 non-null   int32 
 12  os_enc           5000 non-null   int32 
 13  device_enc       5000 non-null   int32 
 14  path_str         5000 non-null   object
 15  query_str        5000 non-null   object
 16  path_seq         5000 non-null   object
 17  query_seq        5000 non-null   

In [391]:
# Store padded sequences into df_nlp
df_nlp['path_seq'] = list(path_seq_padded)
df_nlp['query_seq'] = list(query_seq_padded)

df_nlp = df_nlp[[
    'method_enc',
    'version',
    'browser_enc',
    'os_enc',
    'device_enc',
    'automation',
    'path_seq',
    'query_seq'
]]
df_nlp.head()

,method_enc,version,browser_enc,os_enc,device_enc,automation,path_seq,query_seq
0,0,1.1,24,5,0,0,"[1, 2, 3, 4, 10, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,0,1.1,15,5,0,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,2,1.1,33,5,0,1,"[11, 26, 12, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,0,1.1,24,5,0,0,"[1, 2, 3, 4, 8, 9, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,0,1.1,24,5,0,0,"[1, 2, 3, 4, 7, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


### Embedding with Keras

In this step, we define a Keras model to generate **trainable embeddings** for the following input fields:

- **`path_seq`** and **`query_seq`** (sequences of tokens)
- **`browser_enc`** and **`os_enc`** (label encoded categories)

#### Steps:

- Calculates vocabulary sizes for each input.
- Builds input layers for each field.
- Applies `Embedding` layers (trainable) to learn vector representations.
- Uses `GlobalAveragePooling1D` to reduce sequences to fixed-size vectors.
- Outputs: Dense vector representations (embeddings) for each field.

These learned embeddings will later be extracted and used as part of the final input to the anomaly detection deep learning model.


In [392]:
# Calculate vocabulary sizes from your dataset
path_vocab_size = max([max(seq) for seq in df_nlp['path_seq'] if len(seq) > 0]) + 1
query_vocab_size = max([max(seq) for seq in df_nlp['query_seq'] if len(seq) > 0]) + 1
browser_vocab_size = df_nlp['browser_enc'].nunique()
os_vocab_size = df_nlp['os_enc'].nunique()


In [393]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D

# Input layers
path_input = Input(shape=(10,), name="path_seq")
query_input = Input(shape=(10,), name="query_seq")
browser_input = Input(shape=(1,), name="browser_enc")
os_input = Input(shape=(1,), name="os_enc")

# Embedding layers (trainable)
path_emb = Embedding(input_dim=path_vocab_size, output_dim=20)(path_input)
query_emb = Embedding(input_dim=query_vocab_size, output_dim=20)(query_input)
browser_emb = Embedding(input_dim=browser_vocab_size, output_dim=8)(browser_input)
os_emb = Embedding(input_dim=os_vocab_size, output_dim=4)(os_input)

# Mean pooling
path_vec = GlobalAveragePooling1D()(path_emb)
query_vec = GlobalAveragePooling1D()(query_emb)
browser_vec = GlobalAveragePooling1D()(browser_emb)
os_vec = GlobalAveragePooling1D()(os_emb)

# Define the model
embedding_model = Model(
    inputs=[path_input, query_input, browser_input, os_input],
    outputs=[path_vec, query_vec, browser_vec, os_vec]
)


### Generate Final Input for Deep Learning

- Use the trained embedding model to convert sequences and encoded values into vector form.
- Add these vectors (`path_vec`, `query_vec`, `browser_vec`, `os_vec`) as new columns.
- Keep only the necessary columns:
  - Encoded values: `method_enc`, `version_enc`, `device_enc`, `automation`
  - Embedding vectors

This prepares the final dataset in numerical format, ready to be fed into a deep learning model.


In [394]:
import numpy as np
import pandas as pd

# Prepare model input
path_input = np.array(path_seq_padded)
query_input = np.array(query_seq_padded)
browser_input = np.array(df_nlp['browser_enc']).reshape(-1, 1)
os_input = np.array(df_nlp['os_enc']).reshape(-1, 1)

# Predict from the embedding model
path_vecs, query_vecs, browser_vecs, os_vecs = embedding_model.predict({
    "path_seq": path_input,
    "query_seq": query_input,
    "browser_enc": browser_input,
    "os_enc": os_input
})

# Add predicted vectors as new columns
df_nlp['path_vec'] = list(path_vecs)
df_nlp['query_vec'] = list(query_vecs)
df_nlp['browser_vec'] = list(browser_vecs)
df_nlp['os_vec'] = list(os_vecs)

df_nlp = df_nlp[[
    'method_enc',
    'version',
    'device_enc',
    'automation',
    'path_vec',
    'query_vec',
    'browser_vec',
    'os_vec'
]]

df_nlp.head()


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


,method_enc,version,device_enc,automation,path_vec,query_vec,browser_vec,os_vec
0,0,1.1,0,0,"[-0.018857202, 0.018225487, -0.019675877, 0.01...","[-0.018091546, -0.038462196, -0.016354978, 0.0...","[0.046689246, -0.01992507, -0.0097543225, -0.0...","[0.014252435, -0.0031387806, 0.009779621, -0.0..."
1,0,1.1,0,1,"[-0.039291706, 0.048203114, -0.04886145, 0.014...","[-0.018091546, -0.038462196, -0.016354978, 0.0...","[-0.033379197, -0.048610833, -0.028714478, -0....","[0.014252435, -0.0031387806, 0.009779621, -0.0..."
2,2,1.1,0,1,"[-0.02007721, 0.03518349, -0.038161427, 0.0008...","[-0.021206785, -0.03869045, -0.01606656, 0.010...","[-0.04512911, -0.018605508, 0.042595867, 0.029...","[0.014252435, -0.0031387806, 0.009779621, -0.0..."
3,0,1.1,0,0,"[-0.010923662, 0.01097886, -0.018969774, 0.020...","[-0.018091546, -0.038462196, -0.016354978, 0.0...","[0.046689246, -0.01992507, -0.0097543225, -0.0...","[0.014252435, -0.0031387806, 0.009779621, -0.0..."
4,0,1.1,0,0,"[-0.019300595, 0.016033214, -0.022062387, 0.01...","[-0.018091546, -0.038462196, -0.016354978, 0.0...","[0.046689246, -0.01992507, -0.0097543225, -0.0...","[0.014252435, -0.0031387806, 0.009779621, -0.0..."
